In [7]:
import pandas as pd
import numpy as np
import pickle
import glob
import os

import matplotlib.pyplot as plt
%matplotlib inline
import matplotlib
from scipy.stats import gaussian_kde

import matplotlib as mpl
matplotlib.rcParams['font.size'] = 8.0
matplotlib.rcParams['pdf.fonttype'] = 42
mpl.rcParams['font.family'] = 'Arial'

In [8]:
# input all parameters
Fs = 3e4
TimeHistory = 20e-3 # in seconds
TimePost = 2e-3 # in seconds

bin_size = 0.1
stim_duration = 10.0

In [9]:
DATA_BASE = '../dataFolder/'

filepath_degrees    = glob.glob(DATA_BASE + '4_RawData_Index_Timestamp/Test/*_timeStamps_degrees.pickle')
filepath_notdegrees = glob.glob(DATA_BASE + '4_RawData_Index_Timestamp/Test/*_timeStamps_scaled.pickle')

out_data = './dataFolder/2_Raster_PSTH_Test/'
out_fig  = './dataFolder/Figures_Test/2_Raster_PSTH_Test/'
os.makedirs(out_data, exist_ok=True)
os.makedirs(out_fig, exist_ok=True)

moth_unit = [('m23', 'unit_1'),
             ('m29', 'unit_1'),
             ('m30', 'unit_1'),
             ('m31', 'unit_1'),
             ('m31', 'unit_2'),
             ('m31', 'unit_3'),
             ('m11', 'unit_2'),
             ('m11', 'unit_3')]
print(f'{len(filepath_degrees)} degree-pickles, {len(filepath_notdegrees)} scaled-pickles')
print(f'{len(moth_unit)} (moth, unit) pairs to process')
print()
print('degree pickles:')
for f in sorted(filepath_degrees):
    print(f'  {f}')
print('scaled pickles:')
for f in sorted(filepath_notdegrees):
    print(f'  {f}')

4 degree-pickles, 1 scaled-pickles
8 (moth, unit) pairs to process

degree pickles:
  ../dataFolder/4_RawData_Index_Timestamp/Test/m23_wn2_timeStamps_degrees.pickle
  ../dataFolder/4_RawData_Index_Timestamp/Test/m29_wn2_timeStamps_degrees.pickle
  ../dataFolder/4_RawData_Index_Timestamp/Test/m30_wn1_timeStamps_degrees.pickle
  ../dataFolder/4_RawData_Index_Timestamp/Test/m31_wn2_timeStamps_degrees.pickle
scaled pickles:
  ../dataFolder/4_RawData_Index_Timestamp/Test/m11_wn2_timeStamps_scaled.pickle


In [11]:
for name, unit_name in moth_unit:
    print(f'processing {name} and {unit_name}')
    
    # get the right timestamp file (degrees if available, else scaled)
    filename = [f for f in filepath_degrees if name in f]
    if not filename:
        filename = [f for f in filepath_notdegrees if name in f][0]
    else:
        filename = filename[0]
    
    with open(filename, "rb") as f:
        testout = pickle.load(f)
    all_Units_aligned = testout[0]
    all_Units_prior = testout[1]
    WhiteNoise = testout[2]
    
    # build PSTH for this unit
    raster = all_Units_aligned[unit_name]
    num_repeats = len(raster)
    spikes = np.concatenate(raster)
    edges = np.arange(0, stim_duration + bin_size, bin_size)
    counts, _ = np.histogram(spikes, bins=edges)
    psth = counts / (num_repeats * bin_size)
    
    # KDE at PSTH bin centers (100 points) and at fine grid (1000 points)
    bin_centers = edges[:-1] + bin_size/2
    kde = gaussian_kde(edges[:-1], weights=psth, bw_method=0.06)
    density_at_psth = kde(bin_centers)
    
    x_grid = np.linspace(edges[:-1].min(), edges[-1].max(), 1000)
    density = kde(x_grid)
    
    # save raster
    raster_data = []
    for repeat_idx, spike_arr in enumerate(raster):
        for spike_time in np.atleast_1d(np.array(spike_arr).flatten()):
            raster_data.append({'repeat': repeat_idx, 'spike_time': spike_time})
    pd.DataFrame(raster_data).to_csv(out_data + name + '_' + unit_name + '_raster.csv', index=False)
    
    # save PSTH + KDE at PSTH bin centers
    pd.DataFrame({'bin_center': bin_centers,
                  'firing_rate': psth,
                  'kde_at_bin': density_at_psth}).to_csv(out_data + name + '_' + unit_name + '_psth.csv', index=False)
    
    # save full KDE
    pd.DataFrame({'time': x_grid,
                  'kde': density}).to_csv(out_data + name + '_' + unit_name + '_kde_full.csv', index=False)
    
    # 3-panel figure
    fig, (ax_stim, ax_raster, ax_psth) = plt.subplots(3, 1, figsize=(3.76, 2.71), sharex=True,
                                                       gridspec_kw={'height_ratios': [0.5, 2, 1], 'hspace': 0.15})
    
    ax_stim.plot(np.arange(0, len(WhiteNoise)) * 1/Fs, WhiteNoise, color='black', linewidth=1)
    stim_min = round(WhiteNoise.min(), 1)
    stim_max = round(WhiteNoise.max(), 1)
    ax_stim.set_ylim(stim_min, stim_max)
    ax_stim.set_yticks([stim_min, 0, stim_max])
    ax_stim.set_ylabel('Proboscis\nDeflection\n(degrees)', fontsize=8)
    ax_stim.tick_params(axis='both', labelsize=7)
    
    ax_raster.eventplot(raster, linelengths=0.5, lineoffsets=1, linewidth=1, color='black')
    ax_raster.set_ylabel('Repeats', fontsize=8)
    ax_raster.tick_params(axis='both', labelsize=7)
    
    ax_psth.bar(edges[:-1], psth, width=bin_size, align='edge', edgecolor='black', color='0.3')
    ax_psth.set_ylabel('Firing rate (spikes/s)', fontsize=8)
    ax_psth.set_xlabel('Time (s)', fontsize=8)
    ax_psth.tick_params(axis='both', labelsize=7)
    ax_psth2 = ax_psth.twinx()
    ax_psth2.plot(x_grid, density, color='blue', linewidth=1.5)
    ax_psth2.set_yticks([])
    
    ax_stim.margins(x=0)
    ax_raster.margins(x=0)
    ax_psth.margins(x=0)
    
    plt.subplots_adjust(left=0.22, top=0.95)
    fig.savefig(out_fig + name + '_' + unit_name + '_combined.pdf', bbox_inches='tight', pad_inches=0.1)
    plt.close(fig)

processing m23 and unit_1
processing m29 and unit_1
processing m30 and unit_1
processing m31 and unit_1
processing m31 and unit_2
processing m31 and unit_3
processing m11 and unit_2
processing m11 and unit_3


In [1]:
import os
print(f'cwd: {os.getcwd()}')

cwd: /Users/meghnaanand/Desktop/PiliferAnalysis/Predictions


In [2]:
import os
print('Train/ exists:', os.path.exists('../dataFolder/4_RawData_Index_Timestamp/Train/'))
print('Train/ contents:', os.listdir('../dataFolder/4_RawData_Index_Timestamp/Train/') if os.path.exists('../dataFolder/4_RawData_Index_Timestamp/Train/') else 'N/A')
print()
print('5_1DCompModel_Train/ exists:', os.path.exists('./dataFolder/5_1DCompModel_Train/'))
print('5_1DCompModel_Train/ contents:', os.listdir('./dataFolder/5_1DCompModel_Train/') if os.path.exists('./dataFolder/5_1DCompModel_Train/') else 'N/A')

Train/ exists: True
Train/ contents: ['m23_wn2_timeStamps_degrees.pickle', 'm30_wn1_timeStamps_degrees.pickle', 'm11_wn2_timeStamps_scaled.pickle', 'm29_wn2_timeStamps_degrees.pickle', 'm31_wn2_timeStamps_degrees.pickle']

5_1DCompModel_Train/ exists: True
5_1DCompModel_Train/ contents: ['m11_unit_2_TrainasCSV.csv', 'm31_unit_1_TrainasCSV.csv', 'm11_unit_2_TrainClippedModel.pickle', '.DS_Store', 'm31_unit_1_TrainEncodingModel.pickle', 'm11_unit_3_TrainasCSV.csv', 'm29_unit_1_TrainEncodingModel.pickle', 'm11_unit_3_TrainEncodingModel.pickle', 'm31_unit_2_TrainasCSV.csv', 'm23_unit_1_TrainasCSV.csv', 'Prediction', 'm29_unit_1_TrainasCSV.csv', 'm31_unit_3_TrainasCSV.csv', 'm11_unit_3_TrainClippedModel.pickle', 'm23_unit_1_TrainClippedModel.pickle', 'm31_unit_1_TrainClippedModel.pickle', 'm31_unit_3_TrainEncodingModel.pickle', 'm31_unit_2_TrainEncodingModel.pickle', 'm30_unit_1_TrainEncodingModel.pickle', 'm30_unit_1_TrainasCSV.csv', 'm31_unit_3_TrainClippedModel.pickle', 'm30_unit_1_Train

In [3]:
import os
print(os.listdir('./dataFolder/5_1DCompModel_Train/Prediction/'))

['m29_unit_1_Prediction.pickle', 'm31_unit_2_Prediction.pickle', 'm30_unit_1_Prediction.pickle', 'm31_unit_3_Prediction.pickle', 'm11_unit_2_Prediction.pickle', 'm23_unit_1_Prediction.pickle', 'm11_unit_3_Prediction.pickle', 'm31_unit_1_Prediction.pickle']
